# Colab surrogate pipeline test

**This notebook cannot measure a score, and is not trying to.**

Colab cannot host `gemma-4-31b-it-qat-w4a16-ct` on any tier. Measured from
our own engine log on 4xL4: weights plus overhead take ~26 GB (nearer 19-20
GB on a single GPU without tensor-parallel buffers), and the grader's 32,768
token context needs a further 39 GB of KV cache at 1.21 MB/token.

| Tier | VRAM | Loads weights | Context |
|---|---|---|---|
| free T4 | 16 GB | **no** | - |
| Pro L4 | 22.5 GB | barely | ~2k |
| Pro A100 | 40 GB | yes | ~12-17k of 32,768 |

A100 is the only tier that loads it, and its context falls *below* the
14,336-token compaction threshold, so the agent's memory behaviour would
differ from the grader's. That is not a measurement.

## What this notebook is for

One question is still open after a week: **has the agent ever produced a
patch?** Not a good patch - any patch. No GPU run has reached a served
request, so the bundle, the ADK agent graph, the tool-calling loop and the
patch-extraction path have never been exercised together.

That question does not need a 31B model. It needs *a* model that calls
tools. So this serves a small instruct model under the competition's model
name - `start_server` already aliases the declared name onto whatever is
served, so **the bundle stays byte-identical and still validates**; only the
weights behind the alias change.

Every ledger row written here is marked `surrogate=True`, and
`pool_by_bundle` drops those rows by default. This matters more than it
looks: a surrogate run has the *real* bundle_hash, so without that flag a
4B model's results would pool straight into the reference arm.

## Stages, cheapest first

Each stage is a hard gate. If stage 2 fails, Colab cannot run this at all
and nothing later is worth attempting.

1. GPU and Python version vs the wheelhouse tags (seconds)
2. **Install the harness** - the real unknown (~10 min)
3. Fetch task data and the lab code
4. Serve a surrogate model, confirm it emits a tool call
5. Run 2 tasks and look for a non-empty patch
6. Save everything to Drive

Stage 2 is the one that decides whether this path exists. Run stages 1-2
before caring about the rest.

## Stage 1 - hardware and Python version

In [ ]:
# Stage 1: what hardware did we get, and can the wheelhouse run on it?
import subprocess, sys, json, os

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

try:
    import torch
    n = torch.cuda.device_count()
    print('torch', torch.__version__, 'CUDA devices', n)
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        cap = p.major + p.minor / 10
        gb = p.total_memory / 1024 ** 3
        print('  GPU', i, p.name, 'cap', cap, 'VRAM', round(gb, 1), 'GB')
        if cap < 8.0:
            print('    WARNING cap < 8.0 (Turing or older).')
            print('    W4A16 compressed-tensors kernels need Ampere.')
            print('    Irrelevant for a surrogate in fp16, fatal for the real model.')
except ImportError:
    print('torch not importable yet')

# The competition wheelhouse is built for cp312. A mismatch here is the
# single thing that killed the Kaggle T4 attempt: nine compiled wheels were
# rejected, vLLM's abi3 extension loaded anyway, and then could not resolve
# its torch symbols. Ten seconds here replaced a 9.5-minute failed run.
PYV = sys.version_info
print('python', PYV.major, PYV.minor, PYV.micro)
if (PYV.major, PYV.minor) != (3, 12):
    print()
    print('*' * 70)
    print('MISMATCH: wheelhouse is cp312, this runtime is',
          str(PYV.major) + '.' + str(PYV.minor))
    print('Compiled wheels (vllm, torch ext, tokenizers) will be rejected.')
    print('Options: a Colab runtime with 3.12, or build the harness from')
    print('source without the prebuilt wheels (slow, unverified).')
    print('*' * 70)

## Stage 2 - install the harness

**The gate.** If this fails on a cp312 mismatch, Colab is not a viable
host and nothing below is worth running.

In [ ]:
# Stage 2: the real unknown - can the harness install here at all?
#
# On Kaggle this comes from an attached dataset. On Colab it has to be
# downloaded, which needs Kaggle credentials. Put them in Colab Secrets
# (key icon, left sidebar) as KAGGLE_USERNAME and KAGGLE_KEY, and grant this
# notebook access. Never paste a key into a cell - it persists in the
# notebook's saved output.
import os, sys, subprocess, time
from pathlib import Path

def sh(cmd, timeout=1800, check=True):
    t0 = time.time()
    print('$', ' '.join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True,
                       text=True, timeout=timeout)
    tail = (r.stdout or '')[-3000:]
    if tail:
        print(tail)
    if r.returncode != 0:
        print('STDERR', (r.stderr or '')[-3000:])
        if check:
            raise RuntimeError('exit ' + str(r.returncode) + ' after ' +
                               str(round(time.time() - t0)) + 's')
    print('  ok in', round(time.time() - t0), 's', flush=True)
    return r

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('Kaggle credentials loaded from Colab Secrets')
except Exception as exc:
    raise SystemExit('Set KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets '
                     'and grant this notebook access. (' + str(exc) + ')')

sh([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'])

WHEELS = Path('/content/wheelhouse')
WHEELS.mkdir(parents=True, exist_ok=True)
if not any(WHEELS.glob('*.whl')):
    # 867 MB. The slow step; ~3-5 min on a Colab link.
    sh(['kaggle', 'datasets', 'download', '-d',
        'metric/gemma-4-developer-agent-wheelhouse',
        '-p', str(WHEELS), '--unzip'], timeout=2400)

whl = sorted(p for p in WHEELS.rglob('*.whl'))
print(len(whl), 'wheels')
if not whl:
    raise SystemExit('No wheels found. Check the dataset slug and that the '
                     'download unzipped.')

# Install from the local wheelhouse only, so nothing silently resolves a
# different version from PyPI. --no-index is the whole point.
r = sh([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
        '--find-links', str(WHEELS)] + [str(p) for p in whl],
       timeout=2400, check=False)
if r.returncode != 0:
    print()
    print('Install failed. If the errors mention cp312 vs this runtime, that')
    print('is the Kaggle T4 failure repeating and Colab is not a viable host.')
    raise SystemExit('wheelhouse install failed')

In [ ]:
# Stage 2b: does the harness actually import? A successful pip install is
# not the same thing - the T4 run installed fine and died on import.
import importlib, traceback

ok = True
for mod in ['torch', 'vllm', 'swegemma', 'google.adk', 'adk_submission']:
    try:
        m = importlib.import_module(mod)
        print('ok  ', mod, getattr(m, '__version__', ''))
    except Exception:
        ok = False
        print('FAIL', mod)
        traceback.print_exc()

assert ok, 'harness does not import; stop here and report the traceback'
print()
print('Stage 2 passed: Colab can host the harness.')

## Stage 3 - task data and lab code

In [ ]:
# Stage 3: task data and the lab code.
#
# Competition data is downloaded for our own use, which the rules permit;
# do not re-share it from here.
from pathlib import Path
import sys, os

DATA = Path('/content/data')
LAB = Path('/content/lab')
DATA.mkdir(parents=True, exist_ok=True)

if not (DATA / 'tasks.jsonl').is_file():
    sh(['kaggle', 'competitions', 'download', '-c',
        'gemma-4-developer-agent', '-p', str(DATA)], timeout=2400)
    import zipfile
    for z in DATA.glob('*.zip'):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA)
        print('extracted', z.name)

for p in sorted(DATA.rglob('*'))[:40]:
    print(' ', p.relative_to(DATA))

# The lab code (src/ + scripts/ + bundles/) lives in a private dataset so
# this notebook stays thin and the same code runs here and on Kaggle.
if not LAB.is_dir():
    sh(['kaggle', 'datasets', 'download', '-d',
        'ahmedsherif382/swegemma-lab', '-p', '/content/lab_dl', '--unzip'],
       timeout=1200)
    LAB = Path('/content/lab_dl')

sys.path.insert(0, str(LAB / 'src'))
print('lab at', LAB)
print(sorted(p.name for p in LAB.iterdir()))

## Stage 4 - surrogate model choice

In [ ]:
# Stage 4: serve a surrogate and confirm it emits a tool call.
#
# Model choice matters less than the tool-call format. The runner's parser
# default is gemma4; a Qwen model needs hermes, and the WRONG parser yields
# zero tool calls, which is indistinguishable from a model that refuses to
# act. That failure mode is why --tool-call-parser exists.
SURROGATE = 'Qwen/Qwen3-4B-Instruct-2507'
PARSER = 'hermes'

# fp16, not bfloat16: T4 is Turing and has no bf16. session_runner already
# checks torch.cuda.is_bf16_supported() and falls back, but say it anyway.
import torch
print('bf16 supported:', torch.cuda.is_bf16_supported())
print('serving', SURROGATE, 'with', PARSER, 'parser')
print()
print('A 4B model will localize badly and fix almost nothing. That is fine.')
print('The question is whether a patch comes out the other end at all.')

## Stage 5 - run two tasks

Two tasks, concurrency 1. Enough to answer the question; cheap enough
to abandon.

In [ ]:
# Stage 5: two tasks, surrogate mode.
#
# --surrogate marks every ledger row so it can never pool with a real run.
# --max-model-len is sized from actual VRAM rather than assumed.
import torch, subprocess, sys
from pathlib import Path

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
MAX_LEN = 16384 if vram_gb >= 20 else 8192
print('VRAM', round(vram_gb, 1), 'GB -> max_model_len', MAX_LEN)

RESULTS = Path('/content/results')
RESULTS.mkdir(parents=True, exist_ok=True)

cmd = [sys.executable, str(LAB / 'scripts' / 'session_runner.py'),
       '--lab', str(LAB),
       '--data', str(DATA),
       '--model-path', SURROGATE,
       '--results', str(RESULTS),
       '--variant', 'v00_public_baseline',
       '--surrogate',
       '--tool-call-parser', PARSER,
       '--reasoning-parser', 'deepseek_r1',
       '--max-model-len', str(MAX_LEN),
       '--limit', '2',
       '--concurrency', '1',
       '--max-time-minutes', '30']

# Hard timeout and a hard exit-code check. A failed stage that idles is how
# 12 GPU-hours were lost on Kaggle; subprocess.call does not raise.
r = subprocess.run(cmd, timeout=60 * 75)
assert r.returncode == 0, 'session_runner exited ' + str(r.returncode)

In [ ]:
# Stage 5b: did a patch come out? This is the whole point of the notebook.
import json
from pathlib import Path

jsonl = RESULTS / 'task_results.jsonl'
rows = []
if jsonl.is_file():
    for line in jsonl.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass

print(len(rows), 'task result(s)')
any_patch = False
for row in rows:
    patch = row.get('prediction') or row.get('agent_patch') or ''
    iid = row.get('instance_id', '?')
    print()
    print('---', iid)
    print('  resolved   ', row.get('resolved'))
    print('  tool calls ', row.get('tool_calls') or row.get('tool_calls_used'))
    print('  error      ', (row.get('error') or '')[:300])
    print('  patch bytes', len(patch))
    if patch.strip():
        any_patch = True
        print(patch[:2000])

print()
if any_patch:
    print('=' * 70)
    print('A PATCH WAS PRODUCED. The pipeline works end to end.')
    print('This says nothing about score - a 4B surrogate wrote it.')
    print('=' * 70)
else:
    print('No patch. Check, in order: were tool calls emitted at all (wrong')
    print('parser gives zero), did the agent hit the budget, is the error')
    print('field set. Zero tool calls with no error means the parser.')

## Stage 6 - persist

In [ ]:
# Stage 6: save to Drive. Colab runtimes reset and take everything with them.
from pathlib import Path
import shutil, datetime, json

try:
    from google.colab import drive
    drive.mount('/content/drive')
    dest = Path('/content/drive/MyDrive/swegemma_runs')
except Exception as exc:
    print('Drive not mounted (' + str(exc) + '); saving locally only')
    dest = Path('/content/saved')

stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
out = dest / ('colab-surrogate-' + stamp)
out.mkdir(parents=True, exist_ok=True)

for src in [RESULTS, Path('/content/results/ledger')]:
    if src.is_dir():
        shutil.copytree(src, out / src.name, dirs_exist_ok=True)

(out / 'environment.json').write_text(json.dumps({
    'surrogate_model': SURROGATE,
    'tool_call_parser': PARSER,
    'max_model_len': MAX_LEN,
    'vram_gb': round(vram_gb, 1),
}, indent=2), encoding='utf-8')

print('saved to', out)
for p in sorted(out.rglob('*')):
    print(' ', p.relative_to(out))